In [1]:
!rm -rf models

In [2]:
!mkdir models



In [3]:
!cd models

In [4]:
!wget https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/candy-9.onnx -O candy.onnx
!wget https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/mosaic-9.onnx -O mosaic.onnx
!wget https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/rain-princess-9.onnx -O rain_princess.onnx
!wget https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/udnie-9.onnx -O udnie.onnx
!wget https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/pointilism-9.onnx -O pointilism.onnx

--2025-12-06 14:37:30--  https://github.com/onnx/models/raw/main/validated/vision/style_transfer/fast_neural_style/model/candy-9.onnx
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/onnx/models/main/validated/vision/style_transfer/fast_neural_style/model/candy-9.onnx [following]
--2025-12-06 14:37:30--  https://media.githubusercontent.com/media/onnx/models/main/validated/vision/style_transfer/fast_neural_style/model/candy-9.onnx
Resolving media.githubusercontent.com (media.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to media.githubusercontent.com (media.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6728029 (6.4M) [application/octet-stream]
Saving to: ‘candy.onnx’

candy.onnx          100%[=============

In [5]:
!pip install onnxruntime pillow torchvision numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00


In [6]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 50.8 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist

In [12]:
import gradio as gr
import onnxruntime as ort
import numpy as np
from PIL import Image
from torchvision import transforms
import os

In [14]:
#ErrorHandler

class ErrorHandler:
    @staticmethod
    def check_model(path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"Modèle ONNX introuvable : {path}")

    @staticmethod
    def onnx_error(err):
        raise RuntimeError(f"Erreur lors de l'inférence ONNX : {err}")

In [15]:
#cache des modèles

model_cache = {}

def load_model(style):
    if style in model_cache:
        return model_cache[style]

    model_path = f"models/{style}.onnx"
    ErrorHandler.check_model(model_path)

    try:
        session = ort.InferenceSession(model_path)
        model_cache[style] = session
        return session
    except Exception as e:
        ErrorHandler.onnx_error(e)

In [ ]:
#conversion tensor to image

def tensor_to_image(tensor):
    if isinstance(tensor, np.ndarray):
        tensor = tensor.squeeze(0)
        tensor = np.clip(tensor, 0, 255)
        tensor = tensor.transpose(1, 2, 0).astype("uint8")
    return Image.fromarray(tensor)

In [17]:
#style transfer

def apply_style_transfer(input_img, style_name, target_size=224):
    if input_img is None:
        return None

    if isinstance(input_img, np.ndarray):
        input_img = Image.fromarray(input_img)

    input_img = input_img.convert("RGB")
    original_size = input_img.size

    preprocess = transforms.Compose([
        transforms.Resize((target_size, target_size)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.mul(255))
    ])

    input_tensor = preprocess(input_img).unsqueeze(0)
    input_array = input_tensor.numpy()

    session = load_model(style_name)
    input_name = session.get_inputs()[0].name

    try:
        output_array = session.run(None, {input_name: input_array})[0]
    except Exception as e:
        ErrorHandler.onnx_error(e)

    output_img = tensor_to_image(output_array)
    output_img = output_img.resize(original_size, Image.Resampling.LANCZOS)

    return output_img

In [18]:
#styles disponibles

styles = {
    "Candy": "candy",
    "Mosaic": "mosaic",
    "Rain Princess": "rain_princess",
    "Udnie": "udnie",
    "Pointillism": "pointilism"
}

In [28]:
#csss

custom_css = """

.gradio-container {
    background-color: #f5f3ef !important;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif !important;
}

#component-0 {
    background-color: #f5f3ef !important;
    max-width: 600px !important;
    margin: 0 auto !important;
    padding: 2rem !important;
}


.logo-header {
    text-align: right;
    color: #b8b0a5;
    font-size: 0.9rem;
    font-weight: 600;
    margin-bottom: 2rem;
    letter-spacing: 0.5px;
}


.main-title {
    text-align: center;
    font-size: 4.5rem;
    font-weight: 900;
    line-height: 1.1;
    margin: 1rem 0 0.5rem 0;
    background: linear-gradient(135deg, #FF00A8 0%, #FF6B00 30%, #FFD800 55%, #FF66CC 75%, #FF99E6 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    letter-spacing: -2px;
}


.subtitle {
    text-align: center;
    color: #bfb5a9;
    font-size: 1rem;
    margin-bottom: 2.5rem;
    font-weight: 400;
}


#upload_button {
    background-color: #a8a8a8 !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 1.5rem 2rem !important;
    font-size: 1.2rem !important;
    font-weight: bold !important;
    color: white !important;
    text-align: center !important;
    cursor: pointer !important;
    transition: all 0.3s ease !important;
    margin-bottom: 1rem !important;
    width: 100% !important;
}

#upload_button:hover {
    background-color: #8a8a8a !important;
    transform: scale(1.02) !important;
}


#style_dropdown {
    margin: 1rem 0 2rem 0 !important;
}

.gr-dropdown {
    background-color: #3d4451 !important;
    border: none !important;
    border-radius: 50px !important;
    color: white !important;
    font-weight: 600 !important;
    padding: 0.8rem 1.5rem !important;
    font-size: 1rem !important;
}


.preview-area {
    border: 3px dashed #c7bfb3 !important;
    border-radius: 15px !important;
    background-color: white !important;
    min-height: 350px !important;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    margin: 2rem 0 !important;
    padding: 2rem !important;
}



.gr-image {
    border-radius: 15px !important;
}


.image-placeholder {
    color: #c7bfb3;
    font-size: 2.5rem;
    text-align: center;
    line-height: 1.8;
}


.gr-form label[data-testid="block-label"] {
    display: none !important;
}


.gr-button {
    border-radius: 50px !important;
    font-weight: bold !important;
    transition: all 0.3s ease !important;
}
"""







In [29]:
#fonction wrapper pour Gradio

def process_image(image, style_choice):
    if image is None:
        return None
    style_key = styles[style_choice]
    result = apply_style_transfer(image, style_key)
    return result

In [30]:

#interface Gradio

with gr.Blocks(css=custom_css, title="AI Style Transfer", theme=gr.themes.Soft()) as demo:


    gr.HTML("""
        <div class="logo-header">
            💎AI Style Transfer
        </div>
    """)


    gr.HTML("""
        <div class="main-title">
            Hi !<br/>Welcome
        </div>
    """)


    gr.HTML("""
        <div class="subtitle">
            Im waiting for you, please enter your photo
        </div>
    """)


    with gr.Row():
        input_image = gr.Image(
            label="📤 UPLOAD",
            type="pil",
            sources=["upload", "clipboard"],
            show_label=True,
            container=True,
            elem_id="upload_button"
        )


    with gr.Row():
        style_dropdown = gr.Dropdown(
            choices=list(styles.keys()),
            value="Candy",
            label="Style",
            show_label=True,
            container=True,
            elem_id="style_dropdown",
            interactive=True
        )


    with gr.Row():
        with gr.Column():

            placeholder = gr.HTML("""
                <div class="preview-area">
                    <div class="image-placeholder">
                        • •<br/><br/>
                        📷<br/>
                        <span style="font-size: 1rem;">Your image will appear here</span>
                    </div>
                </div>
            """, visible=True)


            output_image = gr.Image(
                label="",
                type="pil",
                show_label=False,
                container=True,
                show_download_button=True,
                visible=False
            )



    def update_visibility(image, style):
        if image is None:
            return gr.update(visible=True), gr.update(visible=False)
        else:
            result = process_image(image, style)
            return gr.update(visible=False), gr.update(visible=True, value=result)


    input_image.change(
        fn=update_visibility,
        inputs=[input_image, style_dropdown],
        outputs=[placeholder, output_image]
    )

    style_dropdown.change(
        fn=update_visibility,
        inputs=[input_image, style_dropdown],
        outputs=[placeholder, output_image]
    )

/tmp/ipython-input-4043896982.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="AI Style Transfer", theme=gr.themes.Soft()) as demo:
/tmp/ipython-input-4043896982.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="AI Style Transfer", theme=gr.themes.Soft()) as demo:


In [31]:
#lancement

if __name__ == "__main__":
    print("\n" + "="*60)
    print("AI STYLE TRANSFER")
    print("="*60)
    print("Interface Gradio prête!")
    print("Lancement de l'application...")
    print("="*60 + "\n")

    demo.launch(
        share=True,
        debug=False,
        show_error=True,
        quiet=False
    )


AI STYLE TRANSFER
Interface Gradio prête!
Lancement de l'application...

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19d92da147b00f66d1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
